# Step 03 — ALKIS refinement

Refine the raw ALKIS/LoD2 layer into something worth carrying downstream. This
notebook is **structural only** — it changes what the rows and columns are, never
what they mean. Semantics (the AdV function codelist, the activity map, the OSM
POI join) are step 04's job.

| | |
|---|---|
| **Reads** | `data/output/02_alkis_lod2_raw.gpkg` (5.07 GB, `MultiPolygon Z`) |
| **Writes** | `data/output/03_alkis_buildings.gpkg` — **the deliverable** |
| | `data/experimental_extract/alkis_outlines.shp` — throwaway QGIS layer |
| **Needs** | `sqlite3` (stdlib), `pyogrio`, `shapely >= 2` |
| **Runtime** | ~10 min, dominated by one 5 GB 3D read |

## What this notebook is

**The hand-off from 3D to 2D.** It takes 4,891,343 LoD2 *surfaces* and produces
1,385,265 flat building footprints, each carrying the 3D information as numbers
rather than as geometry. Notebook 04 enriches these with OSM.

One row per **LoD2 part** (`gml_id`), not per ALKIS object. Every height and roof
attribute is a property of the part — `height_ridge_m`, `roof_shape`, `ridge_z_m`
all differ between the parts of one building — so dissolving to `alkis_id` would
force an arbitrary winner and lose the rest irreversibly. `alkis_id` rides along
on every row, so aggregating up to real buildings is a one-line `groupby`
whenever a later step wants it.

| section | |
|---|---|
| 1–4 | Column triage: profile all 27, drop the 7 that carry no information |
| 5 | Classify every surface as `GROUND` / `WALL` / `ROOF` using Z |
| 6 | Build the part layer: footprint, both volumes, every surviving attribute |
| 7 | Write `03_alkis_buildings.gpkg` |
| 8 | Experimental: dissolve to ALKIS outlines for QGIS (throwaway) |

Deliberately **not** done here: no volume threshold, no merging of adjacent
polygons, no function-code labels. Those need either the OSM layer or a decision
this notebook has no evidence for.

## Why SQL and not geopandas

The raw layer is 4,891,343 rows and 4.26 GB, most of it geometry. Loading the
attributes into pandas costs GBs of RAM — `Eigentum` and `Lizenz` alone are
~200 bytes on each of 4.9 M rows. A GeoPackage *is* a SQLite database, so the
profile below is one aggregate query against it: no geometry is read and nothing
is materialised in Python.

One pass with all 109 aggregates takes ~70 s. Computing them column by column
instead means 54 full scans of a 4.26 GB file — about 10 minutes — because each
scan pulls the geometry pages too whether you asked for them or not.

In [ ]:
import os, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
# Same reason as steps 01 and 02: a notebook's working directory is not
# necessarily its own folder, so Path('..') is unreliable.
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError(
        'Cannot find the pipeline root (the folder containing config.py). '
        f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.'
    )
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

# --- point GDAL/PROJ at this env's data files ---------------------------------
_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import sqlite3, time, warnings
import pandas as pd
import pyogrio

from config import (
    ALKIS_RAW_FILE, ALKIS_DROP_CONSTANT_COLS, TARGET_CRS,
    EXPERIMENTAL_DIR, ALKIS_OUTLINES_SHP,
    ALKIS_BUILDINGS_FILE, ALKIS_RENAME, ALKIS_OUTPUT_COLS,
)
from lib.checks import require_file, require_non_empty, require_unique

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

print('Root :', ROOT_DIR)
print('Raw  :', ALKIS_RAW_FILE.name)

## 1. Input contract

The raw layer must exist and must still be the thing step 02 described: one row
per LoD2 *surface*, so `gml_id` repeats. Nothing here assumes otherwise.

`mode=ro` on the SQLite connection is deliberate — this notebook must not be
able to modify the source, and read-only also means it can be opened while QGIS
has the layer loaded.

**Read-only does not mean lock-free.** On Windows an open handle blocks
*deletion* of a file even when opened read-only, while the file still reports as
writable throughout. So the connection is closed explicitly at the end of
section 4. Left open, this kernel pins the layer and step 02 can never rewrite
it, failing with `[WinError 32] being used by another process` — which looks
exactly like a QGIS lock and is not one. If a step 02 rerun ever reports that
lock and QGIS is closed, restart this notebook's kernel.

In [ ]:
require_file(ALKIS_RAW_FILE, 'raw ALKIS/LoD2 layer')

info = pyogrio.read_info(ALKIS_RAW_FILE, layer='buildings')
print(f'  ..  rows     : {info["features"]:,}')
print(f'  ..  geometry : {info["geometry_type"]}')
print(f'  ..  CRS      : {info["crs"]}')
if info['crs'] != TARGET_CRS:
    raise AssertionError(f'raw layer is {info["crs"]}, expected {TARGET_CRS}')

# A GeoPackage is a SQLite database. Read-only, so this cannot touch the source
# and does not fight QGIS for the file.
con = sqlite3.connect(f'file:{ALKIS_RAW_FILE}?mode=ro', uri=True)
cur = con.cursor()

schema = list(cur.execute('PRAGMA table_info(buildings)'))
GEOM_COL = 'geom'
ATTRS = [r[1] for r in schema if r[1] not in ('fid', GEOM_COL)]
print(f'  ..  columns  : {len(ATTRS)} attributes + fid + {GEOM_COL}')

N = cur.execute('SELECT COUNT(*) FROM buildings').fetchone()[0]
if N != info['features']:
    raise AssertionError(f'SQLite sees {N:,} rows, GDAL sees {info["features"]:,}')
print(f'  ok  {N:,} surface rows, agreed by SQLite and GDAL')

## 2. Profile every column

For each of the 27 attributes: how many rows are filled, how many distinct
values exist, and the value range. That is 4 aggregates × 27 columns + the row
count = 109 aggregates, all in **one** query so the file is scanned once.

`COUNT(col)` ignores NULLs while `COUNT(*)` does not, which is what makes the
fill rate fall out of the same pass. `COUNT(DISTINCT col)` is the column that
actually decides things here: **1 distinct value means the column cannot
distinguish any building from any other.**

In [ ]:
parts = ['COUNT(*)']
for c in ATTRS:
    parts += [f'COUNT("{c}")', f'COUNT(DISTINCT "{c}")',
              f'MIN("{c}")', f'MAX("{c}")']

print(f'Profiling {len(ATTRS)} columns in one pass over {N:,} rows '
      f'(~70 s) ...', flush=True)
t0 = time.perf_counter()
row = cur.execute('SELECT ' + ', '.join(parts) + ' FROM buildings').fetchone()
print(f'  ok  one pass, {len(parts)} aggregates  [{time.perf_counter() - t0:,.1f}s]')

recs = []
for i, c in enumerate(ATTRS):
    filled, distinct, lo, hi = row[1 + i * 4: 5 + i * 4]
    numeric = isinstance(lo, (int, float)) and isinstance(hi, (int, float))
    recs.append({
        'column':   c,
        'filled':   filled,
        'fill_pct': round(100.0 * filled / N, 1),
        'distinct': distinct,
        'detail':   (f'{lo:,.3f} .. {hi:,.3f}' if numeric else str(lo)[:44]),
    })

profile = pd.DataFrame(recs)
print()
print(profile.to_string(index=False))

## 3. Triage — what goes

**The rule for this step is narrow on purpose: a column goes only if it has
exactly one distinct value across all 4.9 M rows.** That is not a judgement
call, it is arithmetic — such a column cannot tell two buildings apart, so
nothing downstream can ever use it.

Everything else stays for now, including the two that *look* droppable:

* `Name` is only 2.0 % filled, but those ~99 k rows are the named buildings —
  halls, schools, venues. Sparse is not useless; that is exactly the population
  step 04 wants to match against OSM names.
* `Strasse` / `HausNr` are 54 % filled. Half is a real limitation to know about
  before anyone plans address matching, but it is not grounds for dropping.

`ALKIS_DROP_CONSTANT_COLS` in `config.py` records the expected set with a reason
per column. The check below **recomputes it from the data and fails if the two
disagree** — a differing set means the source release changed and the stored
reasoning needs revisiting, rather than a stale list quietly dropping the wrong
thing.

In [ ]:
constant = set(profile.loc[profile['distinct'] <= 1, 'column'])
declared = set(ALKIS_DROP_CONSTANT_COLS)

if constant != declared:
    raise AssertionError(
        'the constant columns found in the data do not match '
        'ALKIS_DROP_CONSTANT_COLS in config.py.\n'
        f'  only in data   : {sorted(constant - declared)}\n'
        f'  only in config : {sorted(declared - constant)}\n'
        'The source release has changed - review the reasons in config.py '
        'before editing the list.'
    )

print(f'  ok  {len(constant)} constant columns, matching config.py exactly')
print()
print('DROP - one distinct value across all rows:')
for c in sorted(constant):
    val = profile.loc[profile['column'] == c, 'detail'].iloc[0]
    print(f'  {c:<12} = {val[:46]:<46}')
    print(f'  {"":<12}   {ALKIS_DROP_CONSTANT_COLS[c]}')

KEEP_COLS = [c for c in ATTRS if c not in constant]
print()
print(f'  ..  columns: {len(ATTRS)} -> {len(KEEP_COLS)}  '
      f'({len(constant)} dropped)')

# What the two bulky ones actually cost. Measured, not guessed: SQLite's
# length() is bytes for text, so this is the on-disk weight of the strings
# themselves, ignoring page overhead.
bulk = [c for c in ('Eigentum', 'Lizenz', 'externRef') if c in ATTRS]
if bulk:
    lens = cur.execute(
        'SELECT ' + ', '.join(f'length("{c}")' for c in bulk) +
        ' FROM buildings LIMIT 1').fetchone()
    print()
    print('  ..  weight of the long repeated strings:')
    total = 0
    for c, ln in zip(bulk, lens):
        mb = (ln or 0) * N / 1e6
        flag = 'DROPPED' if c in constant else 'kept for now'
        total += mb if c in constant else 0
        print(f'        {c:<11} {ln:>4} bytes x {N:,} rows = {mb:>7,.0f} MB   {flag}')
    print(f'        -> dropping the constant ones sheds ~{total:,.0f} MB of the '
          f'{ALKIS_RAW_FILE.stat().st_size / 1e6:,.0f} MB file')

## 4. What survives, and what each column is for

The 20 survivors, grouped by the job they do. This is the map the remaining
steps work from — and the "decide next" group is exactly what steps 03.2 and
03.3 have to resolve.

Value breakdowns are printed for the low-cardinality survivors, because for
those the question "is this useful?" is answerable by just looking at the
values. They are read in a single pass for the same reason as the profile.

One discrepancy to expect rather than puzzle over: `GrundrissA` shows **13**
distinct in the profile above and **14** here. Both are right. SQL's
`COUNT(DISTINCT)` ignores NULL; the breakdown below uses
`value_counts(dropna=False)`, which counts "missing" as its own category. The
extra row is the 72,749 NULLs, and seeing them is the point.

In [ ]:
ROLES = {
    'identity': ['gml_id'],
    'volume':   ['measHeight', 'Firsthoehe', 'Traufhoehe', 'AbsHoehe', 'DachFlaech'],
    'semantic': ['function', 'Name'],
    'location': ['AGS', 'Stadt', 'Strasse', 'HausNr'],
    'decide next': ['externRef', 'creationDa', 'GrundrissA', 'DqDach',
                    'roofType', 'DachName', 'DachNeig', 'DachOri'],
}

seen = set()
for role, cols in ROLES.items():
    here = [c for c in cols if c in KEEP_COLS]
    seen |= set(here)
    print(f'{role.upper()}  ({len(here)})')
    for c in here:
        r = profile.loc[profile['column'] == c].iloc[0]
        print(f'   {c:<12} {r["fill_pct"]:>6.1f}% filled  {r["distinct"]:>9,} distinct  '
              f'{r["detail"][:40]}')
    print()

unclassified = [c for c in KEEP_COLS if c not in seen]
if unclassified:
    print(f'!!  not classified above: {unclassified}')

# Low-cardinality survivors: one scan, then count in pandas.
small = [c for c in KEEP_COLS
         if 1 < profile.loc[profile['column'] == c, 'distinct'].iloc[0] <= 40]
print('=' * 72)
print(f'value breakdown of the {len(small)} low-cardinality survivors: {small}')
print('=' * 72)
t0 = time.perf_counter()
vals = pd.read_sql_query(
    'SELECT ' + ', '.join(f'"{c}"' for c in small) + ' FROM buildings', con)
print(f'  ..  read in one pass  [{time.perf_counter() - t0:,.1f}s]')

for c in small:
    vc = vals[c].value_counts(dropna=False)
    print(f'\n  {c}  ({len(vc)} distinct)')
    for v, k in vc.head(10).items():
        print(f'      {str(v)[:44]:<44} {k:>10,}  {100 * k / N:>5.1f}%')
    if len(vc) > 10:
        print(f'      ... and {len(vc) - 10} more')

# --- release the file ---------------------------------------------------------
# MUST close. An open SQLite handle blocks DELETION of the GeoPackage on Windows
# even when it was opened read-only, and the file still reports as writable the
# whole time. Left open, this kernel pins the multi-GB layer and step 02 can
# never rewrite it - it fails with `[WinError 32] being used by another process`,
# which reads exactly like a QGIS lock and is not one.
con.close()
print()
print('  ok  SQLite connection closed - the raw layer is no longer pinned')

## 5. Classify every surface using Z

Step 02 keeps the Z ordinate, which makes the surfaces **classifiable** rather
than guessable:

| class | test |
|---|---|
| `GROUND` | flat in Z (`zmax − zmin ≤ 1 cm`) **and** sitting at the part's own `AbsHoehe` |
| `WALL` | spans Z **and** projects to ~zero XY area (vertical) |
| `ROOF` | everything else |

**Verified against CityGML's own labels**, which the shapefile export discards.
Across 20 tiles, 10,803 parts, 98,789 surfaces: every `GroundSurface` found
(100.0000 %), nothing else ever classified `GROUND` (0 false positives), and the
footprint areas identical to `+0.000000 %`. `ClosureSurface` — the virtual panes
that seal archways and carport gaps — is vertical, so all 10,591 of them land in
`WALL` and are discarded. For comparison, the original pipeline's
`groupby('gml_id')['area_m2'].idxmax()` is exact for only 99.62 % of parts,
because it picks the roof wherever a roof overhangs the walls.

Two ordering rules the cell depends on:

* **Z statistics come from the original 3D geometry**, and must be taken before
  anything is flattened.
* **Every area comes from a repaired 2D projection.** Flattening leaves 52.5 % of
  surfaces self-intersecting — a hip roof's faces meet at an apex and project
  onto shared edges — and an invalid ring yields both a wrong `.area` and a
  `TopologyException` from `union_all`. The 3D geometry is kept in place
  regardless, because the volume needs the roof faces' Z.

In [ ]:
import geopandas as gpd
import numpy as np
import shapely
from shapely.geometry import Polygon

Z_TOL = 0.01   # metres. LoD2 stores mm precision, so 1 cm is generous.

# Everything the output needs, in one read. This is the expensive step - the
# geometry alone is ~4 GB - so it happens exactly once and both the deliverable
# and the experimental layer are built from it.
_READ = ['gml_id', 'externRef', 'function', 'measHeight',
         'Firsthoehe', 'Traufhoehe', 'AbsHoehe', 'DachFlaech',
         'DachNeig', 'DachOri', 'roofType', 'DachName', 'DqDach',
         'creationDa', 'GrundrissA', 'Name', 'AGS', 'Stadt', 'Strasse', 'HausNr']

print(f'Reading {len(_READ)} columns + 3D geometry from '
      f'{ALKIS_RAW_FILE.name} ({ALKIS_RAW_FILE.stat().st_size / 1e9:,.2f} GB; '
      f'~3-5 min) ...', flush=True)
t0 = time.perf_counter()
surf = gpd.read_file(ALKIS_RAW_FILE, layer='buildings', columns=_READ)
print(f'  ok  {len(surf):,} surface rows  [{time.perf_counter() - t0:,.0f}s]')

if not surf.geometry.has_z.any():
    raise AssertionError(
        'the raw layer has no Z ordinate, so surfaces cannot be classified and '
        'no exact volume is possible. Re-run step 02 with '
        'LOD2_MERGE_FLATTEN_Z = False.'
    )
print('  ok  geometry carries Z')

# alkis_id: rpartition, NEVER str.split('$$$'). pandas treats a >1-char split
# pattern as a regex, and '$$$' is three end-anchors - it matches the empty
# string at the end and every id silently becomes ''.
surf['alkis_id'] = surf['externRef'].str.rpartition('$$$')[2]
surf = surf.drop(columns='externRef')
_bad = ~surf['alkis_id'].fillna('').str.startswith('DENIAL')
if _bad.any():
    raise AssertionError(
        f'{int(_bad.sum()):,} rows carry no DENIAL-prefixed ALKIS id. '
        f'Examples: {surf.loc[_bad, "alkis_id"].head(3).tolist()}'
    )
print(f'  ok  alkis_id on every row, {surf["alkis_id"].nunique():,} distinct')

In [ ]:
# --- Z statistics, from the 3D geometry --------------------------------------
t0 = time.perf_counter()
geoms = surf.geometry.values
coords = shapely.get_coordinates(geoms, include_z=True)
counts = shapely.get_num_coordinates(geoms)
row_of = np.repeat(np.arange(len(surf)), counts)
zc = pd.Series(coords[:, 2])
surf['zmin'] = zc.groupby(row_of).min().values
surf['zmax'] = zc.groupby(row_of).max().values
del coords, row_of, zc

# --- 2D projection, REPAIRED, then measured ----------------------------------
g2d = shapely.force_2d(geoms)
_invalid = ~shapely.is_valid(g2d)
n_invalid = int(_invalid.sum())
if n_invalid:
    g2d[_invalid] = shapely.buffer(g2d[_invalid], 0)
print(f'  ..  repaired {n_invalid:,} of {len(surf):,} projected geometries '
      f'({100 * n_invalid / len(surf):.1f} %)')
surf['geom2d'] = g2d
surf['area_xy'] = shapely.area(g2d)

_flat      = (surf['zmax'] - surf['zmin']) <= Z_TOL
_at_ground = (surf['zmax'] - surf['AbsHoehe']).abs() <= Z_TOL
_vertical  = surf['area_xy'] < 0.01
surf['surface_class'] = np.where(_flat & _at_ground, 'GROUND',
                          np.where((~_flat) & _vertical, 'WALL', 'ROOF'))
print(f'  ok  classified  [{time.perf_counter() - t0:,.0f}s]')
print('  ..  ' + str(surf['surface_class'].value_counts().to_dict()))

# Exactly one GROUND per part is what the whole approach rests on, so it is
# measured on the full region rather than trusted from the tile sweeps.
n_surfaces = surf.groupby('gml_id').size()
per_part = surf[surf['surface_class'] == 'GROUND'].groupby('gml_id').size()
n_parts = len(n_surfaces)
print()
print(f'  ..  parts total          : {n_parts:,}')
print(f'  ..  parts with 1 GROUND  : {int((per_part == 1).sum()):,} '
      f'({100 * (per_part == 1).sum() / n_parts:.3f} %)')
print(f'  ..  parts with >1 GROUND : {int((per_part > 1).sum()):,}')
print(f'  ..  parts with NO GROUND : {n_parts - len(per_part):,}')

_bad_parts = set(n_surfaces.index) - set(per_part[per_part == 1].index)
if _bad_parts:
    _sub = surf[surf['gml_id'].isin(_bad_parts)]
    print(f'  !!  {len(_bad_parts):,} parts cannot yield a footprint and are '
          f'excluded from the output:')
    print(f'        surface rows each : '
          f'{_sub.groupby("gml_id").size().value_counts().sort_index().to_dict()}')
    print(f'        classes present   : {_sub["surface_class"].value_counts().to_dict()}')

## 6. Build the part layer — footprint and both volumes

The geometry becomes the **`GROUND` polygon**, flat, in EPSG:25832. Everything
3D is retained as numbers.

### The two volumes

| column | how it is computed |
|---|---|
| `volume_3d_m3` | Σ over `ROOF` faces of `projected_area × (mean corner z − ground z)` |
| `volume_old_m3` | `area_m2 × height_ridge_m` — the original pipeline's method |
| `volume_ratio` | `volume_old_m3 / volume_3d_m3` |

`volume_3d_m3` is exact for a flat ground and planar roof faces, which is what
LoD2 guarantees. Think of the building as a bundle of vertical columns standing
on the footprint: each roof face contributes its own shadow area times its
average height above the ground. No per-roof-shape formula is needed — gable,
hip, pyramid and barrel all fall out of the same sum.

**The average height is not the mean of the corner heights.** For a plane the
average over an *area* is the height at the **area centroid**, whereas averaging
the corner coordinates gives the **vertex centroid** — and those coincide only
for triangles and parallelograms. An earlier version of this cell took the
vertex mean; measured against the exact integral over 2,201 buildings it was
wrong for **25.1 %** of them, −1.28 % in aggregate but individual buildings off
by up to **6.7×**, concentrated in faces with five or more unevenly spaced
corners. Per-building volume *is* the redistribution weight, so the cell now
fan-triangulates each face with signed 2D areas, which is exact for any planar
polygon, convex or not.

`volume_old_m3` treats the building as a box up to the **ridge**, so it counts
the empty wedge between eaves and ridge as solid:

```
        ╱▔╲
      ╱▒▒▒▒▒╲        ▒ = air that volume_old_m3 counts as building
    ╱─────────╲
    │         │
    └─────────┘
```

Both are kept so the difference against the original pipeline stays auditable,
and `volume_ratio` makes it inspectable per building. The region-wide
overstatement is printed by the cell below — it is exactly 1.000 for flat roofs,
where no wedge exists, and rises with roof pitch.

### Column names

`ALKIS_RENAME` in `config.py` maps the AdV shorthand to readable names. Note the
z/height distinction: `Firsthoehe`, `Traufhoehe` and `AbsHoehe` are **absolute
elevations above sea level**, so they become `ridge_z_m`, `eaves_z_m`,
`ground_z_m`. `measHeight` really is a height above ground — and it is the
ridge, not the eaves — so it becomes `height_ridge_m`.

In [ ]:
def _prism_volume(geom, ground_z):
    # Volume between the ground plane and this face, EXACTLY.
    #
    # The volume under a planar face is  INT z dA - ground_z * area, and for a
    # plane INT z dA equals the area times z at the face's AREA centroid.
    #
    # An earlier version used `area x mean(vertex z)` - the VERTEX centroid -
    # which coincides with the area centroid only for triangles and
    # parallelograms. Measured over 2,201 buildings that shortcut was wrong for
    # 25.1 % of them: -1.28 % in total, but individual buildings off by up to
    # 6.7x, concentrated in faces with 5+ unevenly spaced corners. Per-building
    # volume IS the redistribution weight, so that was not acceptable.
    #
    # Fan triangulation with SIGNED 2D areas is exact for any planar face,
    # convex or not: z at a triangle's centroid is the plain mean of its three
    # vertex heights, and the signed areas make concave parts cancel correctly.
    tot = 0.0
    for poly in getattr(geom, 'geoms', [geom]):
        for ring, sign in ([(poly.exterior, 1.0)] +
                           [(h, -1.0) for h in poly.interiors]):
            c = np.asarray(ring.coords)
            if len(c) > 1 and np.allclose(c[0], c[-1]):
                c = c[:-1]
            if len(c) < 3:
                continue
            x, y, z = c[:, 0], c[:, 1], c[:, 2]
            # signed area of each fan triangle (v0, vi, vi+1)
            sa = 0.5 * ((x[1:-1] - x[0]) * (y[2:] - y[0]) -
                        (x[2:] - x[0]) * (y[1:-1] - y[0]))
            zbar = (z[0] + z[1:-1] + z[2:]) / 3.0
            contrib = float(np.sum(sa * (zbar - ground_z)))
            # Normalise THIS RING's winding, not the running total. The rings of
            # one MultiPolygon are not consistently wound, so summing raw signed
            # contributions makes faces cancel: two identical flat faces with
            # opposite winding gave 0 instead of 800. Deciding the sign per ring
            # keeps the fan cancellation that makes concave faces exact, while
            # every face still adds.
            if float(np.sum(sa)) < 0:
                contrib = -contrib
            tot += sign * contrib
    return tot


usable = per_part[per_part == 1].index
ground = (surf[(surf['surface_class'] == 'GROUND') & surf['gml_id'].isin(usable)]
          .set_index('gml_id'))
print(f'  ..  parts with a usable GROUND footprint: {len(ground):,}')

roof = surf[(surf['surface_class'] == 'ROOF') &
            surf['gml_id'].isin(ground.index)].copy()
roof['_gz'] = roof['gml_id'].map(ground['zmax'])
print(f'  ..  computing volume_3d_m3 over {len(roof):,} roof faces '
      f'(a few minutes) ...', flush=True)
t0 = time.perf_counter()
roof['_v'] = [_prism_volume(g, gz)
              for g, gz in zip(roof.geometry.values, roof['_gz'].values)]
vol3d = roof.groupby('gml_id')['_v'].sum()
n_roof_faces = roof.groupby('gml_id').size()
print(f'  ok  [{time.perf_counter() - t0:,.0f}s]')

# --- assemble -----------------------------------------------------------------
bld = gpd.GeoDataFrame(
    ground.drop(columns=['zmin', 'zmax', 'area_xy', 'surface_class', 'geom2d',
                         ground.geometry.name]),
    geometry=ground['geom2d'].values, crs=ground.crs)
bld['area_m2'] = ground['area_xy'].values

bld = bld.rename(columns=ALKIS_RENAME)
bld['height_eaves_m'] = (bld['eaves_z_m'] - bld['ground_z_m']).round(3)
bld['volume_3d_m3']   = vol3d.reindex(bld.index).fillna(0.0).round(1)
bld['volume_old_m3']  = (bld['area_m2'] * bld['height_ridge_m']).round(1)
bld['volume_ratio']   = (bld['volume_old_m3'] /
                         bld['volume_3d_m3'].replace(0, np.nan)).round(4)
bld['n_surfaces']     = n_surfaces.reindex(bld.index).astype('int32')
bld['n_roof_faces']   = n_roof_faces.reindex(bld.index).fillna(0).astype('int32')
bld['area_m2']        = bld['area_m2'].round(2)
bld = bld.reset_index().rename(columns={'index': 'gml_id'})
if 'gml_id' not in bld.columns:
    bld = bld.rename(columns={bld.columns[0]: 'gml_id'})

missing = [c for c in ALKIS_OUTPUT_COLS if c not in bld.columns]
if missing:
    raise AssertionError(f'ALKIS_OUTPUT_COLS names columns that do not exist: {missing}')
extra = [c for c in bld.columns if c not in ALKIS_OUTPUT_COLS]
if extra:
    print(f'  ..  dropping columns not in ALKIS_OUTPUT_COLS: {extra}')
bld = bld[ALKIS_OUTPUT_COLS]

require_unique(bld, 'gml_id', 'alkis buildings')
print()
print('  ..  the volume comparison, region-wide:')
print(f'        volume_old_m3 total : {bld["volume_old_m3"].sum() / 1e6:>10,.1f} million m3')
print(f'        volume_3d_m3  total : {bld["volume_3d_m3"].sum() / 1e6:>10,.1f} million m3')
print(f'        old overstates by   : '
      f'{100 * (bld["volume_old_m3"].sum() / bld["volume_3d_m3"].sum() - 1):+.2f} %')
print(f'        volume_ratio median / mean / p95: '
      f'{bld["volume_ratio"].median():.3f} / {bld["volume_ratio"].mean():.3f} / '
      f'{bld["volume_ratio"].quantile(.95):.3f}')
print()
print('  ..  by roof shape (median volume_ratio):')
_t = bld.groupby('roof_shape').agg(n=('volume_ratio', 'size'),
                                   ratio=('volume_ratio', 'median'))
print(_t.sort_values('n', ascending=False).head(10).round(3).to_string())

## 7. Write the deliverable

GeoPackage, not shapefile: one file, no 2 GB component cap, no 10-character
field-name truncation, and proper NULLs. The file is unlinked first because a
GeoPackage write *appends*, so a stale layer from an earlier run would otherwise
survive beside the new one.

If this fails with `[WinError 32] being used by another process`, something has
the file open — QGIS, or another notebook's kernel. Note that an open
**read-only** SQLite handle is enough to block deletion on Windows while the file
still reports as writable.

In [ ]:
if ALKIS_BUILDINGS_FILE.exists():
    try:
        ALKIS_BUILDINGS_FILE.unlink()
    except PermissionError as e:
        raise RuntimeError(
            f'{ALKIS_BUILDINGS_FILE.name} is locked by another process, so it '
            'cannot be replaced. QGIS holds a GeoPackage open while the layer is '
            'loaded, and so does another notebook kernel with an open sqlite3 '
            'connection - even a read-only one. Close them and rerun this cell. '
            f'Original error: {e}'
        ) from None

print(f'Writing {len(bld):,} rows x {len(bld.columns)} columns to '
      f'{ALKIS_BUILDINGS_FILE.name} ...', flush=True)
t0 = time.perf_counter()
bld.to_file(ALKIS_BUILDINGS_FILE, layer='buildings', driver='GPKG')
print(f'  ok  {ALKIS_BUILDINGS_FILE.stat().st_size / 1e6:,.1f} MB  '
      f'[{time.perf_counter() - t0:,.0f}s]')

check = gpd.read_file(ALKIS_BUILDINGS_FILE, layer='buildings', rows=5)
print(f'  ok  read back: CRS {check.crs}, '
      f'geometry {check.geometry.geom_type.unique().tolist()}, 3D {check.geometry.has_z.any()}')
print()
print(check.drop(columns='geometry').head(5).to_string())
print()
print('  ..  columns:')
for c in bld.columns:
    if c == 'geometry':
        continue
    nn = bld[c].notna().sum()
    print(f'        {c:<18} {100 * nn / len(bld):>6.1f} % filled   {str(bld[c].dtype):<10}')

## 8. Experimental — ALKIS object outlines for QGIS

**Throwaway. Nothing reads this.** It dissolves the part footprints to one
polygon per `alkis_id`, which is the shape a person means by "a building", and
lands in `data/experimental_extract/` so nothing downstream can depend on it.

Cheap because it reuses the layer built above: 601,047 of 869,327 ALKIS objects
have exactly one part, so 69 % of the groups are pass-throughs.

`n_parts > 1` is the useful filter — a correct dissolve gives one coherent
outline. The `MultiPolygon` count is the same signal numerically: an outline that
comes out multi-part has pieces that do not touch.

In [ ]:
EXPERIMENTAL_DIR.mkdir(parents=True, exist_ok=True)

parts = bld[['gml_id', 'alkis_id', 'volume_3d_m3', 'volume_old_m3', 'geometry']].copy()
parts['is_uuid'] = parts['gml_id'].str.startswith('UUID_')
agg = parts.groupby('alkis_id').agg(
    n_parts=('gml_id', 'size'),
    n_uuid=('is_uuid', 'sum'),
    vol_3d=('volume_3d_m3', 'sum'),
    vol_old=('volume_old_m3', 'sum'),
)

t0 = time.perf_counter()
print(f'  ..  dissolving {len(parts):,} footprints by alkis_id ...', flush=True)
outlines = parts[['alkis_id', 'geometry']].dissolve(by='alkis_id').reset_index()
print(f'  ok  {len(outlines):,} outlines  [{time.perf_counter() - t0:,.0f}s]')

outlines = outlines.merge(agg, on='alkis_id', how='left')
outlines['area_m2']   = outlines.geometry.area.round(2)
outlines['vol_ratio'] = (outlines['vol_old'] /
                         outlines['vol_3d'].replace(0, np.nan)).round(4)
for c in ('n_parts', 'n_uuid'):
    outlines[c] = outlines[c].astype('int32')
for c in ('vol_3d', 'vol_old'):
    outlines[c] = outlines[c].round(1)
outlines = outlines[['alkis_id', 'n_parts', 'n_uuid', 'area_m2',
                     'vol_3d', 'vol_old', 'vol_ratio', 'geometry']]
require_unique(outlines, 'alkis_id', 'outlines')

print(f'  ..  multi-part outlines (pieces that do not touch): '
      f'{int((outlines.geometry.geom_type == "MultiPolygon").sum()):,} '
      f'({100 * (outlines.geometry.geom_type == "MultiPolygon").mean():.2f} %)')

for ext in ('.shp', '.shx', '.dbf', '.prj', '.cpg'):
    ALKIS_OUTLINES_SHP.with_suffix(ext).unlink(missing_ok=True)
outlines.to_file(ALKIS_OUTLINES_SHP, driver='ESRI Shapefile')
_tot = sum(p.stat().st_size for p in
           ALKIS_OUTLINES_SHP.parent.glob(ALKIS_OUTLINES_SHP.stem + '.*'))
print(f'  ok  {ALKIS_OUTLINES_SHP.name} + sidecars, {_tot / 1e6:,.1f} MB')

## 9. Where this leaves us

**`03_alkis_buildings.gpkg`** — 1,385,265 flat footprints, one per LoD2 part,
each with `alkis_id` for traceability back to the cadastre and up to real
buildings. This is the input to notebook 04.

What was established getting here:

* `gml_id` is a **part** key (1,385,279), not a building key. `externRef`'s `$$$`
  tail is the ALKIS object id, present on 100 % of rows for both the `DENILD…`
  (81 %) and `UUID…` (19 %) forms, at 1.59 parts per object, max 228.
* Surfaces are **classified by Z**, verified against CityGML's own labels on
  10,803 parts: every `GroundSurface` found, zero false positives, footprint
  areas identical to `+0.000000 %`.
* **`volume_old_m3` overstates volume** and the bias is *directional* — exact
  for flat roofs, tens of percent for pitched. Flat roofs skew commercial and industrial
  while gables skew residential, so the original method inflates residential
  stock by roughly a quarter relative to commercial. In a volume-proportional
  redistribution that misplaces workers and retail demand into housing; it
  changes *where* demand lands, not merely the scale.

Still open, deliberately:

* **no volume threshold.** The distribution is in the output; pick the number
  from it rather than inheriting `>= 1 m³`.
* **no merging** of touching or overlapping polygons.
* **no function labels.** `function` is the raw AdV code. Joining
  `building_function_codelist_de_en.csv` (301 codes) and
  `alkis_building_activity_map.xlsx` (280) belongs in notebook 04 — and the
  question that matters there is whether all 88 codes occurring in this region
  appear in the activity map, since a code missing there yields no activities and
  those buildings would drop out of the redistribution silently.